In [101]:
import time
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
from scipy.optimize import minimize



# 1. Market-Based Stock Scoring and Initial Portfolio Construction

# 0) Parameter Settings

ann_fac = 252

# Scoring function: dual standardization
# Sharpe weight = 0.7, Alpha weight = 0.3, Beta penalty
wS, wA = 0.7, 0.3
lam = 0.2  # beta penalty strength

# Attack sector definition and allocation constraint
attack_sectors = {"Semiconductors", "Industrials", "Energy"}
attack_target = 0.70  # 70% in attack sectors, 30% in others

# Minimum weight per asset
min_w = 0.05

# Maximum number of stocks per sector
max_per_sector = 2

benchmark = "SPY"


# Last 3 monghts and 10 years window for portfolio optimization
end_3m = pd.Timestamp("2026-04-17")
end_10y = pd.Timestamp("2026-04-17")

start_3m = end_3m - pd.Timedelta(days=92)
start_10y = end_10y - pd.Timedelta(days=365*10 + 10)



# 1) Investment Universe

# 20 Stocks
universe = {
    "Semiconductors": ["TSM", "ASX"],
    "Industrials": ["CAT", "RTX", "GE", "MMM"],
    "Energy": ["XOM", "CVX", "COP", "SHEL"],
    "Materials": ["BHP", "LIN", "NEM", "APD"],
    "ConsumerStaples": ["PG", "HSY"],
    "Healthcare": ["JNJ", "ABBV", "LLY", "UNH"],
}

sector_of = {t: sec for sec, lst in universe.items() for t in lst}
all_candidates = sorted(list(sector_of.keys()))
download_list_3m = sorted(set(all_candidates + [benchmark]))


# 2) Helper Functions

def zscore(s: pd.Series) -> pd.Series:
    s = s.astype(float)
    return (s - s.mean()) / s.std(ddof=0)

def safe_download_close(tickers, start, end, pause_sec=0.2):
    """
    Robust data download:
    auto_adjust=True ensures Close is adjusted for splits/dividends
    """
    data = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False,
        group_by="column"
    )

    # Handle MultiIndex columns
    if isinstance(data.columns, pd.MultiIndex):
        if "Close" not in data.columns.get_level_values(0):
            raise RuntimeError("No Close field returned.")
        px = data["Close"].copy()
    else:
        if "Close" not in data.columns:
            raise RuntimeError("No Close column returned.")
        px = data["Close"].copy()

    px = px.dropna(how="all").dropna(axis=1, how="all")

    time.sleep(pause_sec)
    return px

def get_rf_dtb3_avg(start_3m, end_3m):
    """
    Fetch risk-free rate from FRED (DTB3: 3-Month Treasury Bill)
    Returns annualized rf in decimal form
    """
    fred = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=DTB3")
    date_col = fred.columns[0]
    rate_col = fred.columns[1]

    fred[date_col] = pd.to_datetime(fred[date_col], errors="coerce")
    fred[rate_col] = pd.to_numeric(fred[rate_col], errors="coerce")
    fred = fred.dropna(subset=[date_col, rate_col])

    samp = fred[(fred[date_col] >= start_3m) & (fred[date_col] <= end_3m)]
    if samp.empty:
        raise RuntimeError("DTB3 empty after filtering window.")

    return float(samp[rate_col].mean() / 100.0)

    

# 3) 3-Month Scoring: Sharpe / Alpha / Beta

px_3m = safe_download_close(download_list_3m, start_3m, end_3m)

# Log returns
ret_3m = np.log(px_3m).diff().dropna()

if benchmark not in ret_3m.columns:
    raise RuntimeError("SPY not downloaded successfully for scoring.")

rm = ret_3m[benchmark].dropna()

# Risk-free rate for 3-month scoring window
rf_annual_3m = get_rf_dtb3_avg(start_3m, end_3m)
rf_daily_3m = rf_annual_3m / ann_fac

rows = []
for t in all_candidates:
    if t not in ret_3m.columns:
        continue

    ri = ret_3m[t].dropna()

    df = pd.concat([ri, rm], axis=1, join="inner")
    df.columns = ["Ri", "Rm"]


    if len(df) < 30:
        continue

    # Excess returns
    df["Ri_excess"] = df["Ri"] - rf_daily_3m
    df["Rm_excess"] = df["Rm"] - rf_daily_3m

    # Annualized Sharpe ratio
    sharpe_ann = (df["Ri_excess"].mean() / df["Ri_excess"].std(ddof=1)) * np.sqrt(ann_fac)

    # CAPM regression: excess stock return on excess market return
    X = np.vstack([np.ones(len(df)), df["Rm_excess"].values]).T
    y = df["Ri_excess"].values

    alpha_daily, beta = np.linalg.lstsq(X, y, rcond=None)[0]
    alpha_ann = alpha_daily * ann_fac

    rows.append({
        "Sector": sector_of[t],
        "Ticker": t,
        "Nobs": len(df),
        "Sharpe_Ann": sharpe_ann,
        "Alpha_Ann": alpha_ann,
        "Beta": beta,
        "AbsBeta": abs(beta),
    })

score_df = pd.DataFrame(rows).dropna()

# Dual standardization + beta penalty
score_df["z_Sharpe"] = zscore(score_df["Sharpe_Ann"])
score_df["z_Alpha"] = zscore(score_df["Alpha_Ann"])
score_df["z_AbsBeta"] = zscore(score_df["AbsBeta"])

score_df["Score"] = wS * score_df["z_Sharpe"] + wA * score_df["z_Alpha"] - lam * score_df["z_AbsBeta"]
score_df = score_df.sort_values("Score", ascending=False).reset_index(drop=True)



# 4) Stock Selection

selected = []
sector_count = {}

def can_add(ticker, sector):
    return sector_count.get(sector, 0) < max_per_sector and ticker not in selected

# Select from attack sectors first
for _, r in score_df.iterrows():
    sec = r["Sector"]
    tic = r["Ticker"]

    if sec in attack_sectors and can_add(tic, sec):
        selected.append(tic)
        sector_count[sec] = sector_count.get(sec, 0) + 1

    if len(selected) >= 6:
        break

# Fill remaining slots
for _, r in score_df.iterrows():
    if len(selected) >= 10:
        break

    sec = r["Sector"]
    tic = r["Ticker"]

    if sec not in attack_sectors and can_add(tic, sec):
        selected.append(tic)
        sector_count[sec] = sector_count.get(sec, 0) + 1

# Output selection
sel_df = score_df[score_df["Ticker"].isin(selected)].copy()
sel_df = sel_df.set_index("Ticker").loc[selected].reset_index()

print("\n=== Selected 10 stocks by Score (sector<=2; attack prioritized) ===")
print(sel_df[["Sector","Ticker","Score","Sharpe_Ann","Alpha_Ann","Beta"]].to_string(index=False))



# 5) Markowitz Optimization (10Y)

px_10y = safe_download_close(selected, start_10y, end_10y)

ret_10y = np.log(px_10y).diff().dropna()

mu = ret_10y.mean().values * ann_fac
Sigma = ret_10y.cov().values * ann_fac

rf = get_rf_dtb3_avg(ret_10y.index.min(), ret_10y.index.max())
print(f"\nrf (DTB3 avg over sample) = {rf:.4%}")

cols = list(px_10y.columns)
n = len(cols)

is_attack = np.array([sector_of[t] in attack_sectors for t in cols], dtype=float)

def port_stats(w):
    r = w @ mu
    vol = np.sqrt(w @ Sigma @ w)
    sharpe = (r - rf) / vol if vol > 0 else -1e9
    return r, vol, sharpe

def neg_sharpe(w):
    return -port_stats(w)[2]

constraints = [
    {"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
    {"type": "eq", "fun": lambda w: (w @ is_attack) - attack_target},
]

bounds = [(min_w, 1.0) for _ in range(n)]

# Initial feasible weights
w0 = np.ones(n) / n

res = minimize(neg_sharpe, w0, method="SLSQP", bounds=bounds, constraints=constraints)

w_star = res.x
r, vol, sh = port_stats(w_star)

out = pd.DataFrame({
    "Sector": [sector_of[t] for t in cols],
    "Ticker": cols,
    "Weight": w_star
}).sort_values("Weight", ascending=False)

print("\n=== Max Sharpe Portfolio ===")
print(f"Return={r:.4f}, Vol={vol:.4f}, Sharpe={sh:.4f}")
print(out.round(4).to_string(index=False))


=== Selected 10 stocks by Score (sector<=2; attack prioritized) ===
         Sector Ticker     Score  Sharpe_Ann  Alpha_Ann      Beta
         Energy   SHEL  1.736862    3.512298   0.892637  0.129442
         Energy    COP  1.297272    2.824131   0.847216 -0.472316
 Semiconductors    ASX  1.269547    3.047901   1.385963  1.839798
    Industrials    CAT  0.332929    1.776387   0.643289  1.683356
 Semiconductors    TSM -0.568383    0.549517   0.167163  1.956411
    Industrials    RTX -0.696116   -0.408287  -0.119487  0.505249
      Materials    LIN  1.075002    2.505467   0.487954  0.083019
      Materials    APD  0.698444    1.738219   0.437763  0.107858
     Healthcare    JNJ  0.576388    1.686531   0.251012  0.085442
ConsumerStaples     PG -0.463309   -0.245991  -0.053710  0.115804

rf (DTB3 avg over sample) = 2.2687%

=== Max Sharpe Portfolio ===
Return=0.2028, Vol=0.2035, Sharpe=0.8850
         Sector Ticker  Weight
 Semiconductors    TSM  0.2967
    Industrials    CAT  0.2033
    

In [103]:
# 2. Financial Statement Analysis

import yfinance as yf
import pandas as pd

universe = {
    "Semiconductors": ["TSM", "ASX"],
    "Industrials": ["CAT", "RTX", "GE", "MMM"],
    "Energy": ["XOM", "CVX", "COP", "SHEL"],
    "Materials": ["BHP", "LIN", "NEM", "APD"],
    "ConsumerStaples": ["PG", "HSY"],
    "Healthcare": ["JNJ", "ABBV", "LLY", "UNH"],
}


tickers = [t for lst in universe.values() for t in lst]

rows = []

for t in tickers:
    try:
        stock = yf.Ticker(t)
        info = stock.info

        rows.append({
            "Ticker": t,
            "ROE": info.get("returnOnEquity"),
            "Net Margin": info.get("profitMargins"),
            "Revenue Growth": info.get("revenueGrowth"),
            "Debt/Equity": info.get("debtToEquity"),
            "P/E": info.get("trailingPE"),
            "EV/EBITDA": info.get("enterpriseToEbitda"),
        })

    except Exception as e:
        print(f"Error for {t}: {e}")
        rows.append({
            "Ticker": t,
            "ROE": None,
            "Net Margin": None,
            "Revenue Growth": None,
            "Debt/Equity": None,
            "P/E": None,
            "EV/EBITDA": None,
        })

df = pd.DataFrame(rows)


# save
df.to_excel("fundamentals_6**.xlsx", index=False)
    
print(df)

   Ticker       ROE  Net Margin  Revenue Growth  Debt/Equity        P/E  \
0     TSM   0.36210     0.46505           0.351       17.131  31.857265   
1     ASX   0.11636     0.06300           0.096       70.617  51.981820   
2     CAT   0.43526     0.13144           0.180      206.670  42.201275   
3     RTX   0.10952     0.07598           0.121       59.513  39.521130   
4      GE   0.44692     0.18982           0.176      114.070  37.733250   
5     MMM   0.75501     0.13027           0.020      276.638  25.758333   
6     XOM   0.11081     0.08905          -0.013       18.936  21.824142   
7     CVX   0.07231     0.06661          -0.082       24.323  27.793053   
8     COP   0.12357     0.13252          -0.068       37.828  18.274015   
9    SHEL   0.10194     0.06683          -0.033       43.172  14.634999   
10    BHP   0.24713     0.18973           0.108       52.639  20.012438   
11    LIN   0.17817     0.20297           0.058       70.630  33.668262   
12    NEM   0.22344     0

In [105]:
# 3. Enhanced Scoring Model: Integrating Market Signals and Fundamental Factors

import numpy as np
import pandas as pd

fund_df = df.copy()

# Extreme / abnormal values handling

# ROE: remove clearly distorted values, e.g. ABBV-like cases
fund_df["ROE"] = fund_df["ROE"].apply(
    lambda x: np.nan if isinstance(x, (int, float)) and (x > 2 or x < -1) else x
)

# Net Margin: remove impossible / highly distorted values
fund_df["Net Margin"] = fund_df["Net Margin"].apply(
    lambda x: np.nan if isinstance(x, (int, float)) and (x > 1 or x < -1) else x
)

# Revenue Growth: optional mild cap for extreme outliers
fund_df["Revenue Growth"] = fund_df["Revenue Growth"].apply(
    lambda x: np.nan if isinstance(x, (int, float)) and (x > 2 or x < -1) else x
)

# Debt/Equity, P/E, EV/EBITDA:
# keep numeric, but turn negative or obviously problematic valuation values into NaN
fund_df["P/E"] = fund_df["P/E"].apply(
    lambda x: np.nan if isinstance(x, (int, float)) and x <= 0 else x
)

fund_df["EV/EBITDA"] = fund_df["EV/EBITDA"].apply(
    lambda x: np.nan if isinstance(x, (int, float)) and x <= 0 else x
)

# Missing value handling 
# Fill missing values with cross-sectional median so score can still be computed
for col in ["ROE", "Net Margin", "Revenue Growth", "Debt/Equity", "P/E", "EV/EBITDA"]:
    fund_df[col] = pd.to_numeric(fund_df[col], errors="coerce")
    med = fund_df[col].median()
    fund_df[col] = fund_df[col].fillna(med)

# Robust z-score helper
def zscore_safe(s: pd.Series) -> pd.Series:
    std = s.std(ddof=0)
    if pd.isna(std) or std == 0:
        return pd.Series(0, index=s.index)
    return (s - s.mean()) / std

# Standardize factors
fund_df["z_ROE"] = zscore_safe(fund_df["ROE"])
fund_df["z_Margin"] = zscore_safe(fund_df["Net Margin"])
fund_df["z_Growth"] = zscore_safe(fund_df["Revenue Growth"])

# lower is better for leverage / valuation
fund_df["z_Debt"] = -zscore_safe(fund_df["Debt/Equity"])
fund_df["z_PE"] = -zscore_safe(fund_df["P/E"])
fund_df["z_EVEBITDA"] = -zscore_safe(fund_df["EV/EBITDA"])

# Fundamental Score 
fund_df["FundScore"] = (
    0.30 * fund_df["z_ROE"] +
    0.20 * fund_df["z_Margin"] +
    0.30 * fund_df["z_Growth"] +
    0.10 * fund_df["z_Debt"] +
    0.05 * fund_df["z_PE"] +
    0.05 * fund_df["z_EVEBITDA"]
)

# Merge with existing market-based score_df 

# score_df is your earlier dataframe from Sharpe / Alpha / Beta scoring
score_df = score_df.merge(
    fund_df[["Ticker", "FundScore"]],
    on="Ticker",
    how="left"
)

# Final Score

# market-based score remains primary; fundamentals act as refinement
score_df["FinalScore"] = score_df["Score"] + 0.5 * score_df["FundScore"]

# Sort by final score
score_df = score_df.sort_values("FinalScore", ascending=False).reset_index(drop=True)

# Check result 
print(score_df[[
    "Sector", "Ticker", "Score", "FundScore", "FinalScore",
    "Sharpe_Ann", "Alpha_Ann", "Beta"
]].to_string(index=False))

         Sector Ticker     Score  FundScore  FinalScore  Sharpe_Ann  Alpha_Ann      Beta
         Energy   SHEL  1.736862  -0.533715    1.470005    3.512298   0.892637  0.129442
 Semiconductors    ASX  1.269547  -0.344566    1.097264    3.047901   1.385963  1.839798
         Energy    COP  1.297272  -0.466805    1.063870    2.824131   0.847216 -0.472316
      Materials    LIN  1.075002  -0.096606    1.026699    2.505467   0.487954  0.083019
         Energy    XOM  0.921077  -0.419015    0.711569    2.257028   0.669575 -0.547177
      Materials    BHP  0.568523   0.164523    0.650785    2.071648   0.749957  1.544489
     Healthcare    JNJ  0.576388   0.095123    0.623949    1.686531   0.251012  0.085442
         Energy    CVX  0.780423  -0.701376    0.429735    2.107766   0.524313 -0.521437
    Industrials    CAT  0.332929   0.134935    0.400396    1.776387   0.643289  1.683356
      Materials    APD  0.698444  -0.980359    0.208264    1.738219   0.437763  0.107858
 Semiconductors    TS

In [107]:
# 4. Stock Selection Based on Composite Final Score

selected = []
sector_count = {}

def can_add(ticker, sector):
    return sector_count.get(sector, 0) < max_per_sector and ticker not in selected

# Sort: FinalScore descending
score_df = score_df.sort_values("FinalScore", ascending=False).reset_index(drop=True)

# Step 1: select from attack sectors first
for _, r in score_df.iterrows():
    sec = r["Sector"]
    tic = r["Ticker"]

    if sec in attack_sectors and can_add(tic, sec):
        selected.append(tic)
        sector_count[sec] = sector_count.get(sec, 0) + 1

    # 6 attack stocks
    if len(selected) >= 6:
        break

# Step 2: fill remaining slots from non-attack sectors
for _, r in score_df.iterrows():
    if len(selected) >= 10:
        break

    sec = r["Sector"]
    tic = r["Ticker"]

    if sec not in attack_sectors and can_add(tic, sec):
        selected.append(tic)
        sector_count[sec] = sector_count.get(sec, 0) + 1

# Output selection
sel_df = score_df[score_df["Ticker"].isin(selected)].copy()
sel_df = sel_df.set_index("Ticker").loc[selected].reset_index()

print("\n=== Selected 10 stocks by FinalScore (sector cap applied; attack prioritized) ===")
print(sel_df[[
    "Sector", "Ticker", "Score", "FundScore", "FinalScore",
    "Sharpe_Ann", "Alpha_Ann", "Beta"
]].to_string(index=False))


=== Selected 10 stocks by FinalScore (sector cap applied; attack prioritized) ===
         Sector Ticker     Score  FundScore  FinalScore  Sharpe_Ann  Alpha_Ann      Beta
         Energy   SHEL  1.736862  -0.533715    1.470005    3.512298   0.892637  0.129442
 Semiconductors    ASX  1.269547  -0.344566    1.097264    3.047901   1.385963  1.839798
         Energy    COP  1.297272  -0.466805    1.063870    2.824131   0.847216 -0.472316
    Industrials    CAT  0.332929   0.134935    0.400396    1.776387   0.643289  1.683356
 Semiconductors    TSM -0.568383   1.438727    0.150980    0.549517   0.167163  1.956411
    Industrials    RTX -0.696116  -0.257774   -0.825003   -0.408287  -0.119487  0.505249
      Materials    LIN  1.075002  -0.096606    1.026699    2.505467   0.487954  0.083019
      Materials    BHP  0.568523   0.164523    0.650785    2.071648   0.749957  1.544489
     Healthcare    JNJ  0.576388   0.095123    0.623949    1.686531   0.251012  0.085442
ConsumerStaples     PG -0.4

In [109]:
# 5. Portfolio Optimization Using Markowitz Framework (10-Year Data)

px_10y = safe_download_close(selected, start_10y, end_10y)

ret_10y = np.log(px_10y).diff().dropna()

mu = ret_10y.mean().values * ann_fac
Sigma = ret_10y.cov().values * ann_fac

rf = get_rf_dtb3_avg(ret_10y.index.min(), ret_10y.index.max())
print(f"\nrf (DTB3 avg over sample) = {rf:.4%}")

cols = list(px_10y.columns)
n = len(cols)

is_attack = np.array([sector_of[t] in attack_sectors for t in cols], dtype=float)

def port_stats(w):
    r = w @ mu
    vol = np.sqrt(w @ Sigma @ w)
    sharpe = (r - rf) / vol if vol > 0 else -1e9
    return r, vol, sharpe

def neg_sharpe(w):
    return -port_stats(w)[2]

constraints = [
    {"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
    {"type": "eq", "fun": lambda w: (w @ is_attack) - attack_target},
]

bounds = [(min_w, 1.0) for _ in range(n)]

# Initial feasible weights
w0 = np.ones(n) / n

res = minimize(neg_sharpe, w0, method="SLSQP", bounds=bounds, constraints=constraints)

w_star = res.x
r, vol, sh = port_stats(w_star)

out = pd.DataFrame({
    "Sector": [sector_of[t] for t in cols],
    "Ticker": cols,
    "Weight": w_star
}).sort_values("Weight", ascending=False)

print("\n=== Max Sharpe Portfolio ===")
print(f"Return={r:.4f}, Vol={vol:.4f}, Sharpe={sh:.4f}")
print(out.round(4).to_string(index=False))


rf (DTB3 avg over sample) = 2.2687%

=== Max Sharpe Portfolio ===
Return=0.2071, Vol=0.2073, Sharpe=0.8896
         Sector Ticker  Weight
 Semiconductors    TSM  0.2983
    Industrials    CAT  0.2017
     Healthcare    JNJ  0.1310
      Materials    LIN  0.0690
         Energy   SHEL  0.0500
    Industrials    RTX  0.0500
ConsumerStaples     PG  0.0500
 Semiconductors    ASX  0.0500
      Materials    BHP  0.0500
         Energy    COP  0.0500


In [111]:
# 6. Exporting Results to Excel

with pd.ExcelWriter("Second_Milestone_Sheets**.xlsx") as writer:

    # Sheet 1
    ret_10y.to_excel(writer, sheet_name="StockData")

    # Sheet 2
    score_df.to_excel(writer, sheet_name="Scoring", index=False)

    # Sheet 3
    sel_df.to_excel(writer, sheet_name="Selected", index=False)

    # Sheet 4
    out.to_excel(writer, sheet_name="Portfolio", index=False)

    # Sheet 5
    fund_df.to_excel(writer, sheet_name="Fundamentals", index=False)